# HOI-based digital twins: hands-on

**Workshop on Whole-Brain Models and High-Order Interactions for Brain Health. IFISC, Palma, 19 November 2026.**

This notebook follows the talk step by step, using the code of [`flehue/brain_twins_brainlat`](https://github.com/flehue/brain_twins_brainlat).

1. What we fit: the O-information matrix
2. The model: homeostatic dynamic mean field (HDMF)
3. From firing rates to the simulated O-information matrix
4. Fitting a digital twin. Step 1: global fit
5. Fitting a digital twin. Step 2: regional fit
6. What a twin gives back
7. Linking twins to people: PCEV

**By the end of this notebook you will be able to:**
- simulate the HDMF and inspect its firing rates, its inhibitory weights and its BOLD signals;
- fit a digital twin to the O-information matrix of one person;
- extract the features of a twin and associate them with phenotypes and expotypes using PCEV, in and out of sample.


**Notation.** $G$: global coupling. $\rho$: target firing rate, one value for the whole brain. $\rho_i$: regional target firing rate. $J_i$: inhibitory weight (I → E) of region $i$. $r^E_i$, $r^I_i$: excitatory and inhibitory firing rates. $e\Omega$, $s\Omega$: empirical and simulated O-information matrices. GOF: correlation between the upper triangles of $e\Omega$ and $s\Omega$. Groups: HC, MCI, AD, FTLD.

**Running it.**
- *Google Colab*: open the notebook from the shared Google Drive folder, save a copy (*File → Save a copy in Drive*) and run it. The first code cell downloads the tutorial folder; nothing needs to be installed.
- *Locally*: download the whole tutorial folder (this notebook, `code/` and `data/`), install the packages listed in `requirements.txt`, and run the notebook from inside that folder.

In both cases, the first run also downloads the published features used in section 7 (96 MB) from the model repository.



## 0. Setup

The tutorial folder holds this notebook, `code/` (the modules of [`flehue/brain_twins_brainlat`](https://github.com/flehue/brain_twins_brainlat) used here: the HDMF simulator, the hemodynamic model and the O-information matrix) and `data/`. In Google Colab, the setup downloads the support files from the public GitHub repository, not from the author's editable Drive. Work on your own notebook copy.

In [ ]:
import sys
import urllib.request
from pathlib import Path

REPO_URL = 'https://github.com/rubenherzog/workshop-wbm-hoi-2026.git'
TUTORIAL_SUBDIR = Path('Materials/Ruben_Herzog/HOI_digital_twins_tutorial')
FEATURES_URL = 'https://raw.githubusercontent.com/flehue/brain_twins_brainlat/main/data/derived/model_output_plus_exposome_data_v3.csv'

if 'google.colab' in sys.modules:
    import subprocess
    repo_root = Path('/content/workshop-wbm-hoi-2026')
    if not (repo_root / TUTORIAL_SUBDIR / 'code' / 'calculate_omat_precision.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_root)], check=True)
    BASE = repo_root / TUTORIAL_SUBDIR
else:
    BASE = Path.cwd()  # run the notebook from inside the downloaded tutorial folder
CODE, DATA = BASE / 'code', BASE / 'data'
FEATURES_FILE = DATA / 'model_output_plus_exposome_data_v3.csv'   # published features (section 7)
if not FEATURES_FILE.exists():
    urllib.request.urlretrieve(FEATURES_URL, FEATURES_FILE)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg, signal
from scipy.stats import gamma as gamma_dist
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed

sys.path.insert(0, str(CODE))
import DMF_ISP_numba_EandI as DMF   # HDMF simulator
import BOLDModel as BD              # hemodynamic model
import calculate_omat               # O-information matrix (Gaussian copula), pairwise loop
import calculate_omat_precision     # O-information matrix, closed form (used in this notebook)

RECOMPUTE = False                   # True: run the regional fit of section 5 instead of loading it

N = 90                              # AAL regions
TRIU = np.triu_indices(N, k=1)      # region pairs i < j: upper triangle without the diagonal
OMEGA_LIM = 0.3                     # colour limit of every O-information matrix (nats)
COL = {'E': '#c0392b', 'I': '#1f4e9b', 'J': '#2e7d32', 'static': '#9e9e9e',
       'PCA': '#002F4A', 'PCEV': '#B85741', 'neutral': '#333333'}
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11, 'figure.dpi': 100})


def to_matrix(upper):
    # upper triangle (N(N-1)/2 values) -> symmetric N x N matrix with zero diagonal
    M = np.zeros((N, N))
    M[TRIU] = upper
    return M + M.T


def show_omat(ax, M, title):
    ax.imshow(M, cmap='RdBu_r', vmin=-OMEGA_LIM, vmax=OMEGA_LIM)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=11)


def omega_colorbar(fig, ax, **kw):
    sm = plt.cm.ScalarMappable(norm=plt.Normalize(-OMEGA_LIM, OMEGA_LIM), cmap='RdBu_r')
    fig.colorbar(sm, ax=ax, label='Ω (nats)', **kw)

## 1. What we fit: the O-information matrix

For each pair of regions $i, j$, with $z$ all the remaining regions:

$$\Omega_{ij} = \Omega(x_i, x_j, z) = I(x_i; x_j) - I(x_i; x_j \mid z)$$

$\Omega_{ij} > 0$: the dependence between $x_i$ and $x_j$ disappears once the rest of the brain is known (redundancy). $\Omega_{ij} < 0$: a dependence appears only once the rest of the brain is known (synergy).

Tutorial data: the empirical O-information matrices of 60 participants (20 HC, 20 AD, 20 FTLD), the group structural connectivity (SC) of each diagnosis, and the parameters the paper fitted for each participant.

In [ ]:
SC = np.load(DATA / 'optimized_SC.npz')                                     # SC_HC, SC_AD, SC_FTLD
meta = pd.read_csv(DATA / 'metadata_subsample.csv').astype({'N_MEGA': str}).set_index('N_MEGA')
emp = np.load(DATA / 'empirical_omats_subsample.npz')                       # upper triangle of eΩ, one per participant
examples = {dx: meta.index[meta.Diagnosis == dx][0] for dx in ['HC', 'AD', 'FTLD']}   # one example participant per diagnosis (the first in the metadata), used in sections 1-6

fig, axs = plt.subplots(4, 3, figsize=(10, 13.3), gridspec_kw={'hspace': 0.25, 'wspace': 0.15})
for col, dx in enumerate(['HC', 'AD', 'FTLD']):
    group = meta.index[meta.Diagnosis == dx]   # the participants of this diagnosis
    sc = SC[f'SC_{dx}']   # group SC of the diagnosis
    omega = to_matrix(np.mean([emp[s] for s in group], 0))                    # mean eΩ of the diagnosis
    omega_one = to_matrix(emp[examples[dx]])                                    # eΩ of the example participant
    im = axs[0, col].imshow(sc, cmap='viridis', norm='log', vmin=1e-3, vmax=1.5)
    axs[0, col].set_xticks([]); axs[0, col].set_yticks([]); axs[0, col].set_title(f'{dx} SC', fontsize=11)
    show_omat(axs[1, col], omega, f'{dx} mean eΩ (n = {len(group)})')
    show_omat(axs[2, col], omega_one, f'{dx} participant eΩ')
    links = sc[TRIU] > 0   # pairs with a structural connection (SC > 0)
    axs[3, col].scatter(sc[TRIU][links], omega_one[TRIU][links], s=2, alpha=0.3, color=COL['PCEV'], label='participant')
    axs[3, col].scatter(sc[TRIU][links], omega[TRIU][links], s=2, alpha=0.3, color=COL['neutral'], label='mean')
    axs[3, col].set_xscale('log'); axs[3, col].set_xlim(1e-6, 2); axs[3, col].axhline(0, color=COL['static'], lw=0.8)
    axs[3, col].set_box_aspect(1)
    axs[3, col].set_xlabel('SC'); axs[3, col].set_title(f'{dx}: SC vs Ω', fontsize=11)
    if col:
        axs[3, col].sharey(axs[3, 0]); axs[3, col].tick_params(labelleft=False)
axs[3, 0].set_ylabel('Ω (nats)')
axs[3, 0].legend(loc='upper left', markerscale=4, frameon=False)
fig.colorbar(im, cax=axs[0, 2].inset_axes([1.05, 0, 0.05, 1]), label='SC')
for row in [1, 2]:
    omega_colorbar(fig, None, cax=axs[row, 2].inset_axes([1.05, 0, 0.05, 1]))
plt.show()

**Look at.** Most entries are positive (red): redundancy dominates at the scale of pairs of regions. The blocks along the diagonal are groups of regions that share information; each twin has to reproduce those of its participant.

## 2. The model: homeostatic dynamic mean field (HDMF)

Each region has an excitatory (E) and an inhibitory (I) population; regions are coupled through their E populations by the SC, scaled by $G$. The inhibitory weight $J_i$ is plastic:

$$\frac{dJ_i}{dt} = -\frac{J_i}{\tau_{decay}} + LR\; r^I_i \left(r^E_i - \rho_i\right)$$

When $r^E_i$ is above its target $\rho_i$, $J_i$ grows and inhibition brings the rate down. The settings are those of the paper pipeline (`pipelines/04_regional_fit/optimize_target_EandI.py`); time is in ms, and $LR = 1/\tau_p$ (`DMF.tau_p`).

Two parameters of the plasticity shape the dynamics:
- $\rho$ sets **where** the excitatory rate settles;
- $LR$ sets **how fast** $J$ reacts. A slow $J$ lags behind $r^E$: excitation builds up, the whole network fires a burst, $J$ catches up and silences it, and the cycle repeats (slow waves). A fast $J$ cancels every fluctuation as it arises: activity becomes asynchronous and irregular.

In [ ]:
def configure(sc, G, tmax=720000, plasticity=True, lr=1 / 1.5, dt=1, sigma=0.01):
    # sets the HDMF parameters in the simulator module (DMF reads them as module globals)
    DMF.SC = 0.2 * sc.copy()                     # SC scaled by 0.2, as in the paper
    DMF.nnodes, DMF.G = N, G                     # number of regions; G scales the long-range input G * sum_j C_ij S_j^E
    DMF.tmax, DMF.dt = tmax, dt                  # simulated time and Euler integration step (ms)
    DMF.downsampling_rates = int(round(10 / dt))  # store one sample every 10 ms
    # noise amplitude; plasticity time constant tau_p = 1 / LR; decay time constant of J
    DMF.sigma, DMF.tau_p, DMF.Jdecay = sigma, 1 / lr, 400000 # parameters derived from the paper Mindlin et al 2026
    DMF.model_1, DMF.model_2 = int(plasticity), 1  # plasticity on or off; decay term on
    DMF.update()                                 # recompile the numba functions: they read the globals above at compile time

**Two knobs: $\rho$ and $LR$.** Coupling $G = 2$, three targets ($\rho$ = 3, 6, 10 Hz) and three learning rates ($LR$ = 1, 10 and 1000). Each simulation lasts 40 s; the last 20 s are shown.

**Predict.** Which of the two parameters changes the *shape* of the activity, and which one changes its *level*?

In [ ]:
G_REG, RHOS, LRS, SIGMA_REG = 2, [3, 6, 10], [1, 10, 1000], 0.005   # coupling, targets (Hz), learning rates, noise
T_SIM, T_SHOW = 40000, 20                        # ms simulated; s shown (the last T_SHOW s)
SC_REG = SC['SC_HC']   # HC group SC for every regime simulation


def run_regime(lr, rho):
    # runs in its own process: configure the model there, then simulate
    configure(SC_REG, G_REG, tmax=T_SIM, lr=lr, dt=0.1, sigma=SIGMA_REG)   # dt = 0.1 ms: Euler step small enough for LR = 1000
    DMF.target = rho * np.ones(N)                # same target firing rate rho (Hz) in every region
    rE, rI, _, J = DMF.Sim(return_J=True)        # Euler integration: rates (Hz) and J_GABA, (time, region), one sample every 10 ms
    return np.stack([rE, rI, J])[:, -T_SHOW * 100:]   # (rE, rI, J) x time x region


keys = [(lr, rho) for lr in LRS for rho in RHOS]   # every (LR, rho) combination
results = Parallel(n_jobs=len(keys))(delayed(run_regime)(lr, rho) for lr, rho in keys)   # one process per simulation
regimes = dict(zip(keys, results))   # (LR, rho) -> array (rE, rI, J) x time x region


def plot_regimes(reduce, title):
    # one panel per (LR, rho): rE and rI (left axis), J (right axis); reduce: (time, region) -> (time,)
    t = np.arange(T_SHOW * 100) / 100
    traces = {k: [reduce(x) for x in v] for k, v in regimes.items()}   # one time series per variable: a single region or the mean over regions
    fig, axs = plt.subplots(len(LRS), len(RHOS), figsize=(14, 6.4), sharex=True)
    for i, lr in enumerate(LRS):
        for j, rho in enumerate(RHOS):
            rE, rI, J = traces[lr, rho]
            ax = axs[i, j]
            ax.plot(t, rE, color=COL['E'], lw=0.7, label='$r^E$')
            ax.plot(t, rI, color=COL['I'], lw=0.7, label='$r^I$')
            ax.axhline(rho, color='k', ls='--', lw=1, label='target ρ')
            ax.set_ylim(0, 1.3 * max(rE.max(), rI.max(), rho))       # headroom for the label
            axJ = ax.twinx()
            axJ.plot(t, J, color=COL['J'], lw=1.2, label='$J$')
            pad = 0.1 * (J.max() - J.min()) + 1e-3
            axJ.set_ylim(J.min() - pad, J.max() + pad); axJ.spines['right'].set_visible(True)
            axJ.tick_params(axis='y', colors=COL['J'])
            if j == len(RHOS) - 1:
                axJ.set_ylabel('$J$', color=COL['J'])
            if j == 0:
                ax.set_ylabel(f'LR = {lr}\nfiring rate (Hz)')
            if i == 0:
                ax.set_title(f'ρ = {rho} Hz', fontsize=11)
            if i == len(LRS) - 1:
                ax.set_xlabel('time (s)')
            ax.text(0.02, 0.95, f'mean $r^E$ = {rE.mean():.1f} Hz', transform=ax.transAxes, va='top', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.85))
    handles = axs[0, 0].get_legend_handles_labels()[0] + [plt.Line2D([], [], color=COL['J'], lw=1.2)]
    fig.legend(handles, ['$r^E$', '$r^I$', 'target ρ', '$J$'], loc='upper center', ncol=4, frameon=False,
               bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(title, y=1.04)
    plt.tight_layout(); plt.show()


REGION = 0
plot_regimes(lambda x: x[:, REGION], f'region {REGION}')   # one region

In [ ]:
plot_regimes(lambda x: x.mean(1), 'mean over the 90 regions')   # mean over regions at each time point

Exploring static vs plastic FIC across G

In [ ]:
RHO0 = 2.7   # target firing rate (Hz), the same in every region
Gs = np.arange(0.5, 4.01, 0.5)   # global couplings tested
last = slice(-2000, None)                        # last 20 s of 60 s
DMF.target = RHO0 * np.ones(N)                   # same target rho in every region
configure(SC['SC_HC'], 1.0, tmax=60000)


rate = {'plastic': [], 'static': []}
for G in Gs:
    configure(SC['SC_HC'], G, tmax=60000, plasticity=True)    # HDMF: J_i follows the plasticity rule
    rate['plastic'].append(DMF.Sim()[0][last].mean())        # mean r^E over regions and the last 20 s
    configure(SC['SC_HC'], G, tmax=60000, plasticity=False)   # J_i stays at its initial value
    J_static = 0.75 * DMF.G * DMF.SC.sum(1) + 1      # linear approximation (Herzog et al. 2024): J_i = 0.75 G sum_j C_ij + 1
    rate['static'].append(DMF.Sim(J0=J_static)[0][last].mean())   # simulation with J_i fixed at J_static

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(Gs, rate['plastic'], 'o-', color=COL['J'], label='plastic $J_i$ (HDMF)')
ax.plot(Gs, rate['static'], 'o-', color=COL['static'], label='static $J_i$, set at G = 1')
ax.axhline(RHO0, color='k', ls='--', lw=1)
ax.set_yscale('log'); ax.set_xlabel('global coupling G'); ax.set_ylabel('mean $r^E$ (Hz)'); ax.legend(frameon=False)
plt.show()

**Look at.** The y-axis is logarithmic. With static $J_i$, more coupling means more excitatory input and nothing compensates it. With plastic $J_i$, each region recalibrates its own inhibition.

**Check.** Plastic $J_i$: between 2.8 and 3.8 Hz for every $G$. Static $J_i$: above 20 Hz from $G = 1.5$, and around 100 Hz at $G = 4$.

## 3. From firing rates to the simulated O-information matrix

Excitatory firing rate → hemodynamic model → BOLD → O-information matrix, as in the paper pipeline: 720 s of simulation, band-pass 0.01–0.1 Hz, one BOLD sample per second. `simulate` also returns the firing rates of the last `rates_window` seconds (100 s by default), used in section 6.

In [ ]:
CUT = 12000                                      # drop the first 12 s of firing rates (one sample every 10 ms)
B_FILT, A_FILT = signal.bessel(3, 2 * 1 * np.array([0.01, 0.1]), btype='bandpass')   # 3rd-order Bessel band-pass 0.01-0.1 Hz; cut-offs as 2 * TR * f, TR = 1 s


# one run of the paper pipeline: HDMF -> BOLD -> FC and O-information matrix
def simulate(target, seed=0, rates_window=100):
    DMF.target, DMF.seed = target, seed          # regional targets rho_i (Hz) and noise seed
    rE, rI, _ = DMF.Sim()                        # firing rates (Hz), (time, region), one sample every 10 ms
    rE, rI = rE[CUT // 10:], rI[CUT // 10:]      # drop the transient
    bold = BD.Sim(rE, N, 0.01)[CUT:][::100]      # hemodynamic model driven by r^E (step 0.01 s); drop the first 120 s; TR = 1 s
    bold = signal.filtfilt(B_FILT, A_FILT, bold, axis=0)   # zero-phase band-pass filter
    keep = int(rates_window * 100)                # samples of 10 ms
    return {'rE': rE[-keep:], 'rI': rI[-keep:], 'bold': bold,   # firing rates of the last rates_window s
            'fc': np.corrcoef(bold.T),           # functional connectivity: Pearson correlation between regions
            'omat': calculate_omat_precision.multi_fc(bold)}   # Omega_ij = I(x_i; x_j) - I(x_i; x_j | rest), Gaussian copula


def gof(s_omat, e_upper):
    # goodness of fit: Pearson correlation between the upper triangles of sOmega and eOmega
    return np.corrcoef(s_omat[TRIU], e_upper)[0, 1]


G0 = 1.3                                         # global coupling of this example
configure(SC['SC_HC'], G0)
sim = simulate(RHO0 * np.ones(N))   # same target in every region

fig = plt.figure(figsize=(11, 6))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.5])
ax = fig.add_subplot(grid[0, :])
ax.plot(sim['bold'][:300, :10] + 0.004 * np.arange(10), color='k', lw=0.7)
ax.set_xlabel('time (s)'); ax.set_ylabel('BOLD, 10 regions'); ax.set_yticks([])
ax = fig.add_subplot(grid[1, 0])
im = ax.imshow(sim['fc'], cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks([]); ax.set_yticks([]); ax.set_title('simulated FC', fontsize=11)
fig.colorbar(im, ax=ax, label='Pearson r')
ax = fig.add_subplot(grid[1, 1])
show_omat(ax, sim['omat'], 'simulated sΩ')
omega_colorbar(fig, ax)
plt.tight_layout(); plt.show()

In [ ]:
G_VALUES = [1.0, 2.0]                            # try other values between 1 and 2.5
e_upper = emp[examples['HC']]   # eOmega of the HC example participant: reference for the GOF

fig, axs = plt.subplots(1, len(G_VALUES), figsize=(4 * len(G_VALUES), 3.6))
for ax, G in zip(axs, G_VALUES):
    configure(SC['SC_HC'], G)
    s_omat = simulate(RHO0 * np.ones(N))['omat']   # sOmega at coupling G, same rho in every region
    show_omat(ax, s_omat, f'G = {G}')
    ax.set_xlabel(f'mean Ω = {s_omat[TRIU].mean():.3f}, GOF = {gof(s_omat, e_upper):.2f}')   # mean of sOmega over pairs; GOF against the HC participant
plt.tight_layout(); plt.show()

## 4. Fitting a digital twin. Step 1: global fit

The paper simulated each SC on a grid of $G$ and $\rho$ (51 × 51, one $\rho$ for the whole brain, 25 seeds) and kept, for each participant, the point with the maximum

$$\mathrm{GOF} = \mathrm{corr}\left(e\Omega_{ij},\, s\Omega_{ij}\right), \quad i<j$$

Here we load those optima and check them by simulation, with 1 seed instead of 25, so the GOF comes out somewhat lower than in the paper.

**Predict.** Do all participants end up at the same point of the grid?

In [ ]:
optima = pd.read_csv(DATA / 'global_fit_all.csv')                           # global optimum of every participant in the paper
G_grid, rho_grid = np.linspace(1, 2.5, 51), np.linspace(2, 4, 51)   # grid of the global fit of the paper (51 x 51)
dG, drho = G_grid[1] - G_grid[0], rho_grid[1] - rho_grid[0]               # grid spacing
edges = [np.r_[rho_grid - drho / 2, rho_grid[-1] + drho / 2], np.r_[G_grid - dG / 2, G_grid[-1] + dG / 2]]   # bin edges centred on the grid points
counts, _, _ = np.histogram2d(optima.target_global, optima.G_global, bins=edges)   # participants per (rho, G) grid point

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(np.log10(counts.T + 1), origin='lower', aspect='auto', cmap='Greys',
          extent=[edges[0][0], edges[0][-1], edges[1][0], edges[1][-1]])
for dx, sid in examples.items():
    x, y = meta.loc[sid, ['target_global', 'G_global']]   # global optimum (rho, G) of the example participant
    ax.plot(x, y, '*', ms=15, color='k', mec='white', mew=1.5)
    ax.annotate(dx, (x, y), xytext=(7, 5), textcoords='offset points', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
ax.set_xlabel('target firing rate ρ (Hz)'); ax.set_ylabel('global coupling G')
ax.set_title('optimum of each participant\n(darker: more participants)', fontsize=10)
plt.show()

**Look at.** The optima spread over the whole grid, and $G$ and $\rho$ trade off along a diagonal: a higher coupling can be compensated by a lower target. Even the global fit is already individual. Some optima lie at the edges of the grid ($\rho$ = 2 or 4 Hz).

In [ ]:
N_SEEDS = 1   # noise seeds averaged per participant (25 in the paper)

fig, axs = plt.subplots(2, 3, figsize=(11, 7.2))
for j, (dx, sid) in enumerate(examples.items()):
    row = meta.loc[sid]   # diagnosis, global optimum and GOF of the paper for this participant
    configure(SC[f'SC_{dx}'], row.G_global)      # SC of the participant's diagnosis and its fitted G
    s_omat = np.mean([simulate(row.target_global * np.ones(N), seed)['omat'] for seed in range(N_SEEDS)], 0)   # sOmega at the global optimum (G, rho), mean over noise seeds
    show_omat(axs[0, j], to_matrix(emp[sid]), f'{dx} {sid}: empirical eΩ')
    show_omat(axs[1, j], s_omat, 'simulated sΩ at the optimum')
    axs[1, j].set_xlabel(f'GOF = {gof(s_omat, emp[sid]):.2f} (paper: {row.GOF_global:.2f})')   # GOF of sOmega against the participant's eOmega
plt.tight_layout(); plt.show()

**Check.** GOF of about 0.28 (HC), 0.40 (AD) and 0.34 (FTLD), against 0.47, 0.55 and 0.46 in the paper with 25 seeds.

**Look at.** The large-scale blocks are reproduced, but $s\Omega$ is weaker and smoother than $e\Omega$: a single $\rho$ for the whole brain cannot reproduce regional differences. That is the job of step 2.

## 5. Fitting a digital twin. Step 2: regional fit

**Predict.** If a region is less connected in the simulation than in the data, should its target firing rate go up or down?

Starting from the global optimum, the target of each region moves with the difference between the empirical and the simulated nodal strength (row sum of $\Omega$):

$$\rho_i(t+1) = \rho_i(t) + \varepsilon\left(\sum_j e\Omega_{ij} - \sum_j s\Omega_{ij}(t)\right), \qquad \varepsilon = 0.08$$

A region less connected than in the data raises its target; a region more connected lowers it. Negative targets are set to zero, and the twin is the iteration with the maximum GOF. The paper runs 100 iterations with 25 seeds and $\varepsilon = 0.04$; here 40 iterations with 1 seed, for one HC participant.

In [ ]:
EPS, N_ITER = 0.08, 40 # play with EPS for faster convergence (EPS = 0.08 is a good compromise between speed and stability)


def regional_fit(sid, eps=EPS, n_iter=N_ITER, use_cache=True):
    path = DATA / f'regional_fit_{sid}_eps{eps}_iter{n_iter}.npz'   # cached result of this fit
    if use_cache and path.exists() and not RECOMPUTE:
        return dict(np.load(path))
    row = meta.loc[sid]   # diagnosis and global optimum (G, rho) of the participant
    e_strength = to_matrix(emp[sid]).sum(1)      # nodal strength of eOmega: sum_j eOmega_ij for each region i
    configure(SC[f'SC_{row.Diagnosis}'], row.G_global)   # G from the global fit, kept fixed
    target = row.target_global * np.ones(N)      # start: the global rho in every region
    out = {'targets': np.zeros((n_iter, N)), 'gof': np.zeros(n_iter), 'omats': np.zeros((n_iter, N, N), np.float32)}   # rho_i, GOF and sOmega of every iteration
    for it in range(n_iter):
        s_omat = simulate(target)['omat']        # sOmega with the current regional targets rho_i
        out['targets'][it], out['gof'][it], out['omats'][it] = target, gof(s_omat, emp[sid]), s_omat
        target = np.clip(target + eps * (e_strength - s_omat.sum(1)), 0, None)   # rho_i <- rho_i + eps (sum_j eOmega_ij - sum_j sOmega_ij), rho_i >= 0
    if use_cache:
        np.savez_compressed(path, **out)
    return out


sid = examples['HC']   # the HC example participant
fit = regional_fit(sid)   # loaded from data/ unless RECOMPUTE = True
best = int(fit['gof'].argmax())                  # the twin: the iteration with the highest GOF
e_omat = to_matrix(emp[sid])   # eOmega of the participant
shown_iter = [0, best, N_ITER - 1]   # first, best and last iterations

fig = plt.figure(figsize=(12, 3.4))
show_omat(fig.add_axes([0.0, 0.08, 0.17, 0.75]), e_omat, 'empirical eΩ')
for k, it in enumerate(shown_iter):
    s_omat = fit['omats'][it].copy()
    np.fill_diagonal(s_omat, 0)
    show_omat(fig.add_axes([0.19 + 0.19 * k, 0.08, 0.17, 0.75]), s_omat, f'simulated sΩ, iteration {it}')
ax = fig.add_axes([0.81, 0.2, 0.18, 0.6])
ax.plot(fit['gof'], color=COL['neutral'])
ax.scatter(shown_iter, fit['gof'][shown_iter], color=COL['neutral'], zorder=3)
ax.axvline(best, color=COL['neutral'], ls=':', lw=1)
ax.annotate('twin', (best, fit['gof'][best]), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=9)
ax.set_xlabel('iteration'); ax.set_ylabel('GOF')
plt.show()

## 6. What a twin gives back

The twin is one $G$ and 90 $\rho_i$. Because it is generative, it gives quantities that fMRI does not: six families of regional features, computed from the firing rates of the last 600 s.

| family | definition |
|---|---|
| rate E, rate I | mean excitatory and inhibitory firing rate |
| ent E, ent I | entropy of a gamma distribution fitted to the firing rates |
| EI rate, EI ent | rate E / rate I and ent E / ent I |

**Predict.** The excitatory rates of the twin are not all equal to one value. Why, if the model regulates them towards a target?

In [ ]:
def gamma_entropy(x):
    # differential entropy of a gamma distribution fitted by maximum likelihood, location fixed at 0
    return gamma_dist.entropy(*gamma_dist.fit(x, floc=0))


rho_twin = fit['targets'][best]                  # regional targets of the best iteration (section 5)
configure(SC['SC_HC'], meta.loc[sid, 'G_global'])   # the twin: the participant's G and fitted rho_i
twin = simulate(rho_twin, rates_window=600)      # firing rates of the last 600 s
features = pd.DataFrame({'rate E': twin['rE'].mean(0), 'rate I': twin['rI'].mean(0),   # mean firing rate of each region
                         'ent E': [gamma_entropy(twin['rE'][:, i]) for i in range(N)],
                         'ent I': [gamma_entropy(twin['rI'][:, i]) for i in range(N)]})   # entropy of the rate distribution of each region
features['EI rate'] = features['rate E'] / features['rate I']   # E/I ratios
features['EI ent'] = features['ent E'] / features['ent I']

region, T_TRACE, T_ZOOM = 0, 60, 5               # region shown; s of trace; s of zoom
t = np.arange(len(twin['rE'])) / 100             # s, from the start of the window
fig = plt.figure(figsize=(14, 6.5))
grid = fig.add_gridspec(2, 3)
ax = fig.add_subplot(grid[0, :])
for population, key in [('E', 'rE'), ('I', 'rI')]:
    ax.plot(t[:T_TRACE * 100], twin[key][:T_TRACE * 100, region], color=COL[population], lw=0.5, label=f'$r^{population}$')
ax.axvspan(0, T_ZOOM, color='k', alpha=0.08)
ax.set_xlabel('time (s)'); ax.set_ylabel('firing rate (Hz)'); ax.set_title(f'region {region}: first {T_TRACE} s', fontsize=11)
for line in ax.legend(frameon=False, loc='upper right', ncol=2).get_lines():
    line.set_linewidth(2)
ax = fig.add_subplot(grid[1, :2])
for population, key in [('E', 'rE'), ('I', 'rI')]:
    ax.plot(t[:T_ZOOM * 100], twin[key][:T_ZOOM * 100, region], color=COL[population], lw=0.8)
ax.set_xlabel('time (s)'); ax.set_ylabel('firing rate (Hz)'); ax.set_title(f'zoom: first {T_ZOOM} s', fontsize=11)
ax = fig.add_subplot(grid[1, 2])
for population, key in [('E', 'rE'), ('I', 'rI')]:
    x = twin[key][:, region]   # firing rates of the region over the 600 s
    shape, loc, scale = gamma_dist.fit(x, floc=0)   # maximum-likelihood gamma fit, location fixed at 0 (rates are positive); the same fit as in gamma_entropy
    xs = np.linspace(0, np.percentile(x, 99.9), 200)   # rates at which the fitted density is evaluated, up to the 99.9th percentile
    ax.hist(x, bins=np.linspace(0, xs[-1], 80), density=True, color=COL[population], alpha=0.35)
    ax.plot(xs, gamma_dist.pdf(xs, shape, loc, scale), color=COL[population], label=f'$r^{population}$, gamma fit')   # fitted gamma density
ax.set_xlabel('firing rate (Hz)'); ax.set_ylabel('density'); ax.set_title(f'600 s, {len(x)} samples', fontsize=11)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**Look at.** The firing rates fluctuate around a stable level for the whole 600 s: the twin is in a stationary regime, so a single distribution summarizes each region. Both distributions are skewed to the right, and the gamma fit captures that shape; its entropy measures how irregular a region is.

**The features of the twin, region by region.** One row per family; the dots in the first row are the fitted targets $\rho_i$.

In [ ]:
fig, axs = plt.subplots(len(features.columns), 1, figsize=(16, 13), sharex=True)
for ax, family in zip(axs, features.columns):
    color = COL['neutral'] if family.startswith('EI') else COL[family[-1]]
    ax.bar(np.arange(N), features[family], width=0.8, color=color)
    ax.axhline(features[family].mean(), color='k', ls='--', lw=0.8)
    lo, hi = features[family].min(), features[family].max()
    if family == 'rate E':
        ax.scatter(np.arange(N), rho_twin, s=10, color='k', zorder=3, label='fitted ρᵢ')   # fitted targets rho_i, to compare with the excitatory rates
        ax.legend(frameon=False, loc='lower right', bbox_to_anchor=(1, 1), borderaxespad=0)
        lo, hi = min(lo, rho_twin.min()), max(hi, rho_twin.max())
    pad = 0.1 * (hi - lo)
    ax.set_ylim(max(lo - pad, 0) if lo > 0 else lo - pad, hi + pad)   # zoom on the range of the values
    ax.set_ylabel(family)
    ax.grid(axis='x', color='0.9', lw=0.5); ax.set_axisbelow(True)
axs[-1].set_xticks(np.arange(N), np.arange(N), fontsize=7, rotation=90)
axs[-1].set_xlim(-0.6, N - 0.4); axs[-1].set_xlabel('region (AAL index)')
plt.tight_layout(); plt.show()

**Look at.** Rate E follows the fitted targets $\rho_i$ but in a compressed range: regions with a high target (e.g. 88, $\rho_i \approx 11.7$ Hz) fire below it, and regions with a target near 0 (e.g. 80) still fire at about 2 Hz, sustained by their inputs. Rate I is not regulated directly, yet it follows rate E, because the inhibitory population is driven by the local excitatory one. The entropies follow the rates: a region with a higher mean rate also has a wider distribution. The six rows are therefore strongly correlated with each other; these 540 values (6 families × 90 regions) are what a twin contributes to the analyses of section 7.

## 7. Linking twins to people: PCEV

PCEV (principal component of explained variance) looks for the weights $w$ over the features whose score $z = Yw$ has the largest fraction of variance explained by the predictors $X$, after removing the confounders $C$:

$$Y = X B^{\mathsf{T}} + C\,\Gamma^{\mathsf{T}} + E \qquad \mathrm{Var}(Y \mid C) = V_M + V_R$$

$$h^2(w) = \frac{w^{\mathsf{T}} V_M\, w}{w^{\mathsf{T}} (V_M + V_R)\, w} \qquad V_M\, w = \lambda\,(V_M + V_R)\, w$$

| symbol | meaning |
|---|---|
| $Y$ | model features (participants × features) |
| $X$ | phenotype or expotype (one or several columns) |
| $C$ | confounders |
| $V_M$, $V_R$ | variance of $Y$ explained by $X$, and residual variance |
| $w$, $\lambda$ | leading eigenvector (the PCEV axis) and its $h^2$ |

PCA finds the direction of largest intrinsic variance of $Y$ (the leading eigenvector of $V_M + V_R$); PCEV finds the direction of largest extrinsic variance, the part explained by $X$.

In [ ]:
def residualize(A, C):
    # centre A and remove the part explained by the confounders C
    A = A - A.mean(0)
    Q, _ = linalg.qr(C, mode='economic')         # orthonormal basis of the columns of C
    return A - Q @ (Q.T @ A)                     # project A onto the complement of C: residuals of the regression of A on C


def pcev_axis(Yr, Xr, ridge=1e-6):
    # Yr, Xr: residualized features and predictors. Returns the PCEV axis w and its h2
    n, p = Yr.shape
    V_total = Yr.T @ Yr / (n - 1) + ridge * np.eye(p)               # V_M + V_R
    S_yx = Yr.T @ Xr / (n - 1)                                      # covariance between features and predictors
    V_M = S_yx @ np.linalg.pinv(Xr.T @ Xr / (n - 1)) @ S_yx.T       # variance of Y explained by X
    lam, W = linalg.eigh(V_M, V_total)                              # V_M w = λ (V_M + V_R) w
    return W[:, -1], lam[-1]                                        # largest lambda = h2 = w' V_M w / w' (V_M + V_R) w


def pcev(Y, X, C):
    Yr, Xr = residualize(Y, C), residualize(X, C)  # remove the confounders from features and predictors
    w, h2_w = pcev_axis(Yr, Xr)   # PCEV axis and its in-sample h2
    return w, h2_w, Yr, Xr


def h2(z, Xr):
    # fraction of the variance of the score z explained by X
    beta = np.linalg.lstsq(Xr, z, rcond=None)[0]                   # least-squares regression of z on X
    return 1 - np.sum((z - Xr @ beta) ** 2) / np.sum(z ** 2)       # R^2 (z has zero mean)


def orient(z, reference):
    # the sign of an eigenvector is arbitrary: make z increase with the reference variable
    return z * np.sign(np.corrcoef(z, reference)[0, 1])

**The data.** The published features of every participant: six families × 90 regions = 540 features per twin. Three analyses, each with the confounders of the paper (the variable under study is left out of the confounders):

| analysis | $X$ | $C$ |
|---|---|---|
| age | age | country, sex, diagnosis, GOF, fMRI data quality (ODQ) |
| diagnosis | one column per group: HC, MCI, AD, FTLD | country, sex, age, GOF, ODQ |
| temperature | the six temperature variables of the exposome | country, sex, diagnosis, age, GOF, ODQ |

The variables of each exposome domain are listed in `data/tutorial/exposome_domains.csv`.

PCEV is first fitted on all participants at once, so $h^2$ is in-sample and optimistic. The last part of this section estimates it out of sample, as in the paper.

In [ ]:
FAMILIES = ['rate_E', 'rate_I', 'ent_E', 'ent_I', 'EI_rate', 'EI_ent']   # six feature families x 90 regions = 540 features (columns <family>_<region>)
DOMAINS = pd.read_csv(DATA / 'exposome_domains.csv').groupby('domain', sort=False).column.apply(list).to_dict()   # exposome domain -> its variables
GROUPS = ['HC', 'MCI', 'AD', 'FTLD']
COVARIATES = ['Age', 'Sex', 'Diagnosis', 'Country', 'gof_corr', 'ODQ_fMRI']   # used as predictors or confounders

FEATURES = [c for c in pd.read_csv(FEATURES_FILE, nrows=0).columns if any(c.startswith(f + '_') for f in FAMILIES)]   # the 540 model features
EXPOSOME = [c for columns in DOMAINS.values() for c in columns]   # every exposome variable
cohort = pd.read_csv(FEATURES_FILE, usecols=['N_MEGA'] + COVARIATES + EXPOSOME + FEATURES, low_memory=False)
cohort['Diagnosis'] = cohort.Diagnosis.replace({'CN': 'HC', 'FTD': 'FTLD'})   # repository labels -> paper labels
cohort = cohort.dropna(subset=COVARIATES + FEATURES).reset_index(drop=True)   # participants with every covariate and feature


def confounders(df, categorical, numeric):
    # confounder matrix C: one-hot columns of the categorical variables + the numeric variables
    return np.c_[pd.get_dummies(df[categorical].astype(str), dtype=float).to_numpy(), df[numeric].to_numpy(float)]


def design(df, X, categorical, numeric):
    # Y: 540 model features; X: predictors; C: confounders
    return {'df': df, 'Y': df[FEATURES].to_numpy(float), 'X': np.asarray(X, float).reshape(len(df), -1),
            'C': confounders(df, categorical, numeric)}


def exposome_design(domain):
    df = cohort.dropna(subset=DOMAINS[domain]).reset_index(drop=True)   # participants with every variable of the domain
    return design(df, df[DOMAINS[domain]], ['Country', 'Sex', 'Diagnosis'], ['Age', 'gof_corr', 'ODQ_fMRI'])


ANALYSES = {
    'age': design(cohort, cohort.Age, ['Country', 'Sex', 'Diagnosis'], ['gof_corr', 'ODQ_fMRI']),   # X: age
    'diagnosis': design(cohort, pd.get_dummies(cohort.Diagnosis)[GROUPS], ['Country', 'Sex'], ['Age', 'gof_corr', 'ODQ_fMRI']),   # X: one column per diagnosis
    'temperature': exposome_design('Temperature'),   # X: the temperature variables of the exposome
}
for name, a in ANALYSES.items():
    print(f'{name}: {len(a["Y"])} participants, {a["Y"].shape[1]} features, {a["X"].shape[1]} predictor(s)')


def violins(ax, z, diagnosis, ylabel):
    parts = ax.violinplot([z[diagnosis == g] for g in GROUPS], showmedians=True)
    for body in parts['bodies']:
        body.set_facecolor(COL['PCEV']); body.set_alpha(0.4)
    for key in ['cbars', 'cmins', 'cmaxes', 'cmedians']:
        parts[key].set_color(COL['PCEV'])
    ax.set_xticks(range(1, len(GROUPS) + 1), GROUPS); ax.set_ylabel(ylabel)

### Age

The same features projected on the PCA axis and on the PCEV axis.

**Predict.** Which of the two scores will follow age?

In [ ]:
a = ANALYSES['age']
w_pcev, h2_pcev, Yr, Xr = pcev(a['Y'], a['X'], a['C'])   # PCEV axis for age and its in-sample h2
w_pca = linalg.eigh(Yr.T @ Yr)[1][:, -1]                            # leading eigenvector of the total variance
scores = {'PCA': orient(Yr @ w_pca, Xr[:, 0]), 'PCEV': orient(Yr @ w_pcev, Xr[:, 0])}   # score of each participant on each axis, z = Yr w

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
for ax, (method, z) in zip(axs, scores.items()):
    ax.scatter(cohort.Age, z, s=2, alpha=0.3, color=COL[method])
    ax.set_title(f'{method} score: h² = {h2(z, Xr):.2f}', color=COL[method], fontsize=11)   # h2 of each score: fraction of its variance explained by age
    ax.set_xlabel('age (years)'); ax.set_ylabel(rf'$z = Y\,w_{{\mathrm{{{method}}}}}$')
plt.tight_layout(); plt.show()

### Diagnosis

$X$ has one column per group, so PCEV looks for the axis that best separates the four groups at once.

**Predict.** Which group will separate most from the others?

In [ ]:
a = ANALYSES['diagnosis']
w, h2_w, Yr, Xr = pcev(a['Y'], a['X'], a['C'])   # PCEV axis for diagnosis (one predictor per group) and its in-sample h2
z = orient(Yr @ w, (cohort.Diagnosis == 'FTLD').to_numpy(float))   # PCEV score, sign set to increase with FTLD

fig, ax = plt.subplots(figsize=(6, 3.8))
violins(ax, z, cohort.Diagnosis, r'$z = Y\,w_{\mathrm{PCEV}}$')
ax.set_title(f'PCEV score: h² = {h2_w:.2f}', fontsize=11)
plt.show()

### Expotype: temperature

With several predictors, the association is between the score and the best linear combination of the predictors.

In [ ]:
a = ANALYSES['temperature']
w, h2_w, Yr, Xr = pcev(a['Y'], a['X'], a['C'])   # PCEV axis for the temperature variables and its in-sample h2
z = Yr @ w                                                          # PCEV score
x_best = Xr @ np.linalg.lstsq(Xr, z, rcond=None)[0]                 # best linear combination of the temperature variables

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.scatter(x_best, z, s=2, alpha=0.3, color=COL['PCEV'])
ax.set_xlabel('temperature variables, best linear combination'); ax.set_ylabel(r'$z = Y\,w_{\mathrm{PCEV}}$')
ax.set_title(f'PCEV score: h² = {h2_w:.2f}', fontsize=11)
plt.show()

**Check.** $h^2$ = 0.14, weaker than for age. The exposures are country-level values and country is a confounder, so only their variation within countries is left.

### Out of sample: cross-validation, as in the paper

In-sample $h^2$ is optimistic, because $w$ is chosen on the same participants it is evaluated on. The paper estimates $h^2$ with repeated stratified 5-fold cross-validation:

1. Split the participants into 5 folds that keep the joint proportions of diagnosis and sex (stratification). Repeat with a new random split.
2. In each fold, learn everything on the training participants only: the means, the regression on the confounders, and the PCEV axis $w$.
3. Apply the training parameters to the held-out participants and project them: $z_{test} = Y_{test}\, w_{train}$.
4. Out-of-sample $h^2$: the fraction of the variance of $z_{test}$ explained by $X_{test}$. Each participant gets one out-of-sample score per repeat; the scores are averaged over repeats.

The paper uses 100 repeats; here 10 (about 30 s).

In [ ]:
N_REPEATS, N_FOLDS, SEED = 10, 5, 2025            # repeated stratified k-fold cross-validation


def fit_residualizer(Y, X, C):
    # means and confounder regressions, learned on the training participants
    r = {'Y': Y.mean(0), 'X': X.mean(0), 'C': C.mean(0)}   # training means
    r['beta_Y'] = np.linalg.lstsq(C - r['C'], Y - r['Y'], rcond=None)[0]   # regression of the features on the confounders
    r['beta_X'] = np.linalg.lstsq(C - r['C'], X - r['X'], rcond=None)[0]   # regression of the predictors on the confounders
    return r


def apply_residualizer(r, Y, X, C):
    # remove the training means and confounder effects; nothing is estimated on the participants passed here
    Cc = C - r['C']   # confounders centred with the training mean
    return (Y - r['Y']) - Cc @ r['beta_Y'], (X - r['X']) - Cc @ r['beta_X']   # residuals of features and predictors


def pcev_cv(Y, X, C, strata):
    # returns the out-of-sample h2 of every fold and the out-of-sample score of every participant (mean over repeats)
    h2_folds, z_test, w_ref = [], np.zeros((N_REPEATS, len(Y))), None   # h2 of each fold; test score of each participant in each repeat
    for rep in range(N_REPEATS):
        folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + rep).split(Y, strata)   # new split in every repeat
        for train, test in folds:
            r = fit_residualizer(Y[train], X[train], C[train])   # confounder model from the training fold only
            w, _ = pcev_axis(*apply_residualizer(r, Y[train], X[train], C[train]))   # PCEV axis from the training fold
            w_ref = w if w_ref is None else w_ref    # reference: the axis of the first fold
            w = w * np.sign(w @ w_ref)                              # same sign of the axis in every fold
            Y_test, X_test = apply_residualizer(r, Y[test], X[test], C[test])   # test fold, residualized with the training parameters
            z_test[rep, test] = Y_test @ w           # out-of-sample scores
            h2_folds.append(h2(Y_test @ w, X_test))  # out-of-sample h2 of the fold
    return np.array(h2_folds), z_test.mean(0)   # each participant is in exactly one test fold per repeat


cv = {}
for name, a in ANALYSES.items():
    strata = (a['df'].Diagnosis + '_' + a['df'].Sex.astype(str)).to_numpy()   # diagnosis × sex
    h2_folds, z_out = pcev_cv(a['Y'], a['X'], a['C'], strata)   # out-of-sample h2 of every fold and out-of-sample scores
    cv[name] = {'in': pcev(a['Y'], a['X'], a['C'])[1], 'out': h2_folds, 'z': z_out}   # in: h2 fitted and evaluated on all participants

fig, ax = plt.subplots(figsize=(6, 3.6))
jitter = np.random.default_rng(0)
for k, (name, res) in enumerate(cv.items()):
    ax.scatter(k + jitter.uniform(-0.12, 0.12, len(res['out'])), res['out'], s=10, alpha=0.6, color=COL['PCEV'],
               label='out of sample (each fold)' if k == 0 else None)
    ax.plot([k - 0.25, k + 0.25], [res['in']] * 2, color='k', lw=2, label='in sample' if k == 0 else None)
ax.set_xticks(range(len(cv)), list(cv)); ax.set_ylabel('h²'); ax.set_ylim(0, 1); ax.legend(frameon=False)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].scatter(cohort.Age, orient(cv['age']['z'], cohort.Age), s=2, alpha=0.3, color=COL['PCEV'])   # out-of-sample score, sign set to increase with age
axs[0].set_xlabel('age (years)'); axs[0].set_ylabel(r'out-of-sample $z = Y\,w_{\mathrm{PCEV}}$')
violins(axs[1], orient(cv['diagnosis']['z'], (cohort.Diagnosis == 'FTLD').to_numpy(float)), cohort.Diagnosis,   # out-of-sample score, sign set to increase with FTLD
        r'out-of-sample $z = Y\,w_{\mathrm{PCEV}}$')
plt.tight_layout(); plt.show()

### Activity: the exposome

Takes one exposome domain, fits PCEV in and out of sample, and reports both $h^2$. Compare the domains at the end: which ones keep an association out of sample?

In [ ]:
# print(list(DOMAINS))
# DOMAIN = 'Air pollution'                         # each group picks a different domain
# a = exposome_design(DOMAIN)
# strata = (a['df'].Diagnosis + '_' + a['df'].Sex.astype(str)).to_numpy()
# h2_in = pcev(a['Y'], a['X'], a['C'])[1]
# h2_out = pcev_cv(a['Y'], a['X'], a['C'], strata)[0]
# print(f'{DOMAIN}: {a["X"].shape[1]} variables, {len(a["Y"])} participants')
# print(f'h² in sample = {h2_in:.2f}; out of sample = {h2_out.mean():.2f} ± {h2_out.std():.2f}')